In [2]:
import fastf1 as f1
import pandas as pd

In [ ]:
session=f1.get_session(2025,'Monza','R')
session.load()

In [4]:
results = session.results

In [5]:
results.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,4781D7,red_bull,Max,Verstappen,Max Verstappen,...,1.0,1,1.0,NaT,NaT,NaT,0 days 01:13:24.325000,Finished,25.0,53.0
4,4,L NORRIS,NOR,norris,McLaren,F47600,mclaren,Lando,Norris,Lando Norris,...,2.0,2,2.0,NaT,NaT,NaT,0 days 00:00:19.207000,Finished,18.0,53.0
81,81,O PIASTRI,PIA,piastri,McLaren,F47600,mclaren,Oscar,Piastri,Oscar Piastri,...,3.0,3,3.0,NaT,NaT,NaT,0 days 00:00:21.351000,Finished,15.0,53.0
16,16,C LECLERC,LEC,leclerc,Ferrari,ED1131,ferrari,Charles,Leclerc,Charles Leclerc,...,4.0,4,4.0,NaT,NaT,NaT,0 days 00:00:25.624000,Finished,12.0,53.0
63,63,G RUSSELL,RUS,russell,Mercedes,00D7B6,mercedes,George,Russell,George Russell,...,5.0,5,5.0,NaT,NaT,NaT,0 days 00:00:32.881000,Finished,10.0,53.0


In [7]:
schedule = f1.get_event_schedule(2026)

In [15]:
schedule.head()

,RoundNumber,Country,Location,OfficialEventName,EventDate,EventName,EventFormat,Session1,Session1Date,Session1DateUtc,...,Session3,Session3Date,Session3DateUtc,Session4,Session4Date,Session4DateUtc,Session5,Session5Date,Session5DateUtc,F1ApiSupport
0,0,Bahrain,Bahrain,FORMULA 1 ARAMCO PRE-SEASON TESTING 1 2026,2026-02-13,Pre-Season Testing,testing,Practice 1,2026-02-11 10:00:00+03:00,2026-02-11 07:00:00,...,Practice 3,2026-02-13 10:00:00+03:00,2026-02-13 07:00:00,None,NaT,NaT,None,NaT,NaT,True
1,0,Bahrain,Bahrain,FORMULA 1 ARAMCO PRE-SEASON TESTING 2 2026,2026-02-20,Pre-Season Testing,testing,Practice 1,2026-02-18 10:00:00+03:00,2026-02-18 07:00:00,...,Practice 3,2026-02-20 10:00:00+03:00,2026-02-20 07:00:00,None,NaT,NaT,None,NaT,NaT,True
2,1,Australia,Melbourne,FORMULA 1 QATAR AIRWAYS AUSTRALIAN GRAND PRIX ...,2026-03-08,Australian Grand Prix,conventional,Practice 1,2026-03-06 12:30:00+11:00,2026-03-06 01:30:00,...,Practice 3,2026-03-07 12:30:00+11:00,2026-03-07 01:30:00,Qualifying,2026-03-07 16:00:00+11:00,2026-03-07 05:00:00,Race,2026-03-08 15:00:00+11:00,2026-03-08 04:00:00,True
3,2,China,Shanghai,FORMULA 1 HEINEKEN CHINESE GRAND PRIX 2026,2026-03-15,Chinese Grand Prix,sprint_qualifying,Practice 1,2026-03-13 11:30:00+08:00,2026-03-13 03:30:00,...,Sprint,2026-03-14 11:00:00+08:00,2026-03-14 03:00:00,Qualifying,2026-03-14 15:00:00+08:00,2026-03-14 07:00:00,Race,2026-03-15 15:00:00+08:00,2026-03-15 07:00:00,True
4,3,Japan,Suzuka,FORMULA 1 ARAMCO JAPANESE GRAND PRIX 2026,2026-03-29,Japanese Grand Prix,conventional,Practice 1,2026-03-27 11:30:00+09:00,2026-03-27 02:30:00,...,Practice 3,2026-03-28 11:30:00+09:00,2026-03-28 02:30:00,Qualifying,2026-03-28 15:00:00+09:00,2026-03-28 06:00:00,Race,2026-03-29 14:00:00+09:00,2026-03-29 05:00:00,True


In [16]:
today = pd.Timestamp.now(tz='UTC')
print(today)

2026-05-19 16:45:16.891948+00:00


In [17]:
completed_races = schedule[
    schedule['Session5Date'] < today
]

In [19]:
last_3_races=completed_races.tail(3)

In [ ]:
from datetime import datetime
all_results = []
year = datetime.now().year

for _, race in last_3_races.iterrows():

    session = f1.get_session(
        year,
        race['EventName'],
        'R'
    )

    session.load()

    results = session.results[
        ['Abbreviation', 'Points']
    ].copy()

    all_results.append(results)

In [25]:
combined_df = pd.concat(all_results)

# Average points of last 3 races
avg_points_df = (
    combined_df
    .groupby('Abbreviation', as_index=False)['Points']
    .mean()
)

# Rename columns
avg_points_df.columns = ['driver', 'points']

# Sort by avg points
avg_points_df = avg_points_df.sort_values(
    by='points',
    ascending=False
)



In [26]:
avg_points_df.head()

,driver,points
2,ANT,25.000000
18,RUS,14.000000
17,PIA,11.000000
9,HAM,10.333333
12,LEC,10.333333


In [47]:
import requests
url="https://api.jolpi.ca/ergast/f1/2026/constructorstandings.json"

response = requests.get(url)
data = response.json()
constructors = data['MRData']['StandingsTable']['StandingsLists'][0]['ConstructorStandings']
df = pd.json_normalize(constructors)

In [48]:
df.head(10)

,position,positionText,points,wins,Constructor.constructorId,Constructor.url,Constructor.name,Constructor.nationality
0,1,1,180,4,mercedes,https://en.wikipedia.org/wiki/Mercedes-Benz_in...,Mercedes,German
1,2,2,110,0,ferrari,https://en.wikipedia.org/wiki/Scuderia_Ferrari,Ferrari,Italian
2,3,3,94,0,mclaren,https://en.wikipedia.org/wiki/McLaren,McLaren,British
3,4,4,30,0,red_bull,https://en.wikipedia.org/wiki/Red_Bull_Racing,Red Bull,Austrian
4,5,5,23,0,alpine,https://en.wikipedia.org/wiki/Alpine_F1_Team,Alpine F1 Team,French
5,6,6,18,0,haas,https://en.wikipedia.org/wiki/Haas_F1_Team,Haas F1 Team,American
6,7,7,14,0,rb,https://en.wikipedia.org/wiki/Racing_Bulls,RB F1 Team,Italian
7,8,8,5,0,williams,https://en.wikipedia.org/wiki/Williams_Racing,Williams,British
8,9,9,2,0,audi,https://en.wikipedia.org/wiki/Audi_in_Formula_One,Audi,German
9,10,10,0,0,cadillac,https://en.wikipedia.org/wiki/Cadillac_in_Form...,Cadillac F1 Team,American


In [49]:
df=df[['position','points','Constructor.name','Constructor.constructorId']]

In [50]:
df.head(10)

,position,points,Constructor.name,Constructor.constructorId
0,1,180,Mercedes,mercedes
1,2,110,Ferrari,ferrari
2,3,94,McLaren,mclaren
3,4,30,Red Bull,red_bull
4,5,23,Alpine F1 Team,alpine
5,6,18,Haas F1 Team,haas
6,7,14,RB F1 Team,rb
7,8,5,Williams,williams
8,9,2,Audi,audi
9,10,0,Cadillac F1 Team,cadillac


In [ ]:
import pandas as pd 
import requests

position_data = requests.get(
    f"https://api.openf1.org/v1/position?session_key={session_key}"
).json()

lap_data = requests.get(
    f"https://api.openf1.org/v1/laps?session_key={session_key}"
).json()

result_data = requests.get(
    f"https://api.openf1.org/v1/session_result?session_key={session_key}"
).json()